# DS ROM dam break – GPU fast smoothing

Same setup as DS_ROM_damBreak: load snapshots, project on grid. We keep the **reference** path (CPU smoothing + `get_predictions`) and add a **fast** path using GPU separable smoothing and `get_predictions_gpu`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter
from datetime import datetime
from ezyrb import POD, RBF, Database, Linear, RegularGrid
from ezyrb import ReducedOrderModel as ROM
from smooth_POD_ROM.reduced_order_model import train_ROM_rw, get_predictions
from smooth_POD_ROM.post_processing import richardson_lucy, richardson_lucy_gpu, get_sigma
import cupy as cp
from cupyx.scipy.ndimage import convolve1d as convolve1d_gpu
from cupyx.scipy.ndimage import gaussian_filter as gaussian_filter_gpu

In [2]:
pth_local_dbCoarseOLD = "C:/Users/florianma/OneDrive - Institutt for Energiteknikk/Documents/convolution_paper/data/DamBreak/"
timestep = 0.25
ts = timestep

n_x = n_y = 256
x = np.linspace(0, 0.584, n_x, endpoint=True)
y = np.linspace(0, 0.584, n_y, endpoint=True)
X, Y = np.meshgrid(x, y, indexing="ij")
is_wall = (0.292 <= X) & (X <= 0.316) & (Y <= 0.048)

case = {
    "sigma": 0.013,
    "c": 1,
    "num_iter": 10,
    "shape": X.shape,
    "x": x,
    "dx": x[1] - x[0],
    "mode": "reflect",
}
# For GPU separable we need truncate (same as CPU loop below)
case_gpu = {**case, "truncate": 10}

In [3]:
X_train = np.load(pth_local_dbCoarseOLD + "t{:.2f}_X_train.npy".format(ts)).T
X_test = np.load(pth_local_dbCoarseOLD + "t{:.2f}_X_test.npy".format(ts)).T
mu_train = np.load(pth_local_dbCoarseOLD + "t{:.2f}_mu_train.npy".format(ts))
mu_test = np.load(pth_local_dbCoarseOLD + "t{:.2f}_mu_test.npy".format(ts))
points = np.load(pth_local_dbCoarseOLD + "points.npy")
print(points.shape, X_train.shape, X_test.shape, mu_train.shape, mu_test.shape)

(36705, 2) (80, 36705) (20, 36705) (80, 2) (20, 2)


In [4]:
X_train_grid = np.empty((len(X_train), n_x * n_y))
X_test_grid = np.empty((len(X_test), n_x * n_y))
for u_set, u_set_grid in zip([X_train, X_test], [X_train_grid, X_test_grid]):
    gridded = griddata(points[:, :2], u_set.T, (X, Y), method="linear")
    failed = np.isnan(gridded)
    gridded[failed] = griddata(points[:, :2], u_set.T, (X, Y), method="nearest")[failed]
    u_set_grid[:] = gridded.reshape((n_x * n_y, len(u_set))).T
gridded_shape = (n_x, n_y)

## Reference path (CPU smoothing + get_predictions)

Smooth with `gaussian_filter` in a loop, train ROM/sROM, then `get_predictions` and apply `is_wall` mask. We keep `X_test_sROMs` as reference.

In [5]:
sgm = case["sigma"] / case["dx"]
X_train_grid_s = np.empty_like(X_train_grid)
for j in range(len(X_train_grid)):
    ss2D = X_train_grid[j].reshape(case["shape"])
    X_train_grid_s[j] = gaussian_filter(ss2D, sigma=sgm, truncate=10, mode='reflect').ravel()

my_ROM = train_ROM_rw(mu_train, X_train_grid)
my_sROM = train_ROM_rw(mu_train, X_train_grid_s)

X_test_ROM = my_ROM.predict(mu_test).snapshots_matrix
t0 = datetime.now()
X_test_sROM, X_test_sROMs = get_predictions(my_sROM, mu_test, **case)
t1 = datetime.now()
print("get_predictions:", (t1 - t0).total_seconds(), "s")
for j in range(len(X_test_sROMs)):
    rmw = X_test_sROMs[j].reshape(gridded_shape)
    rmw[is_wall] = 0.5
    X_test_sROMs[j] = rmw.ravel()

X_train_grid_s_ref = X_train_grid_s.copy()  # reference for comparison
X_test_sROM_ref = X_test_sROM.copy()  # reference for comparison
X_test_sROMs_ref = X_test_sROMs.copy()  # reference for comparison

get_predictions: 1.97148 s


## GPU separable smoothing + get_predictions_gpu

Winner: `smooth_snapshots_gpu_separable`. We define it and `get_predictions_gpu`, then run the fast path.

In [6]:
def _smooth_gpu2_compute(imgs, sgm, truncate, mode):
    return gaussian_filter_gpu(imgs, sigma=(0, sgm, sgm), truncate=truncate, mode=mode)


def smooth_snapshots_gpu2(X, case):
    shape = case["shape"]
    sgm = case["sigma"] / case["dx"]
    truncate = case["truncate"]
    imgs = cp.asarray(X.reshape(-1, *shape).astype(np.float32))
    mode = case["mode"]
    out = _smooth_gpu2_compute(imgs, sgm, truncate, mode)
    return cp.asnumpy(out).reshape(len(X), -1)


def smooth_snapshots_gpu_separable(X, case):
    """GPU: batched separable 1D Gaussian convolutions (horizontal then vertical)."""
    sgm = case["sigma"] / case["dx"]
    shape = case["shape"]
    truncate = case["truncate"]
    radius = int(truncate * sgm + 0.5)
    x_k = cp.arange(-radius, radius + 1, dtype=cp.float32)
    kernel = cp.exp(-(x_k ** 2) / (2 * sgm ** 2))
    kernel /= kernel.sum()
    imgs = cp.asarray(X.reshape(-1, *shape).astype(np.float32))
    tmp = convolve1d_gpu(imgs, kernel, axis=2, mode=case["mode"])
    out = convolve1d_gpu(tmp, kernel, axis=1, mode=case["mode"])
    return cp.asnumpy(out).reshape(len(X), -1)


def get_predictions_gpu(sROM, mu_, sigma, c, num_iter, shape, x, sROM_only=False, monitor_progress_postprocessing=False, monitor_convergence=False, sigmaD="calc_based_on_distance", **kwargs):
    mu_ = np.asarray(mu_)
    X_test_sROM = sROM.predict(mu_).snapshots_matrix
    mu_train = sROM.database.parameters_matrix
    data = X_test_sROM
    if sROM_only:
        return data, None
    mode = kwargs.get("mode", "wrap")

    if monitor_convergence:
        deconvolved = np.ones((*data.shape, num_iter))
    else:
        deconvolved = np.empty_like(data)
    # Transfer all data to GPU once
    data_gpu = cp.asarray(data.astype(np.float64))
    for j in range(len(mu_)):
        if isinstance(sigmaD, np.ndarray):
            sgm_est = sigmaD[j]
        elif sigmaD == "calc_based_on_distance":
            sgm_est = get_sigma(sigma, mu_[j][None, ...], mu_train, c=c)
        else:
            raise ValueError("unknown method for sigmaD.")
        res = richardson_lucy_gpu(
            x,
            data_gpu[j].reshape(shape),
            sgm_est,
            num_iter,
            mode=mode,
            monitor_convergence=monitor_convergence,
        )[0].reshape(deconvolved[j].shape)
        deconvolved[j] = res
        if monitor_progress_postprocessing:
            print(j, end=", ")
    X_test_sROMs = deconvolved
    return X_test_sROM, X_test_sROMs

In [7]:
# Fast path: GPU separable smoothing + same train + get_predictions_gpu
_ = smooth_snapshots_gpu2(X_train_grid[:1], case_gpu)  # warm-up
cp.cuda.Stream.null.synchronize()

X_train_grid_s_fast = smooth_snapshots_gpu2(X_train_grid, case_gpu)
print("same?", np.allclose(X_train_grid_s, X_train_grid_s_fast))
cp.cuda.Stream.null.synchronize()

my_sROM2 = train_ROM_rw(mu_train, X_train_grid_s_fast)
t0 = datetime.now()
X_test_sROM_ref, X_test_sROMs_ref = get_predictions(my_sROM, mu_test, **case)
t1 = datetime.now()
X_test_sROM_fast, X_test_sROMs_fast= get_predictions_gpu(my_sROM, mu_test, **case)
t2 = datetime.now()
print("get_predictions:", (t1 - t0).total_seconds(), "s")
print("get_predictions_gpu:", (t2 - t1).total_seconds(), "s")
print("same X_test_sROM?", np.allclose(X_test_sROM_ref, X_test_sROM_fast))
print("same X_test_sROMs?", np.allclose(X_test_sROMs_ref, X_test_sROMs_fast))



for j in range(len(X_test_sROMs_fast)):
    rmw = X_test_sROMs_fast[j].reshape(gridded_shape)
    rmw[is_wall] = 0.5
    X_test_sROMs_fast[j] = rmw.ravel()
print("same X_test_sROM?", np.allclose(X_test_sROM_ref, X_test_sROM_fast))
print("same X_test_sROMs?", np.allclose(X_test_sROMs_ref, X_test_sROMs_fast))

same? True
get_predictions: 1.730695 s
get_predictions_gpu: 0.525852 s
same X_test_sROM? True
same X_test_sROMs? True
same X_test_sROM? True
same X_test_sROMs? False


In [8]:
# Compare reference vs fast path (only smoothing changed; RL is same CPU)
diff = np.abs(X_test_sROMs_ref - X_test_sROMs_fast)
print("same output?", np.allclose(X_test_sROMs_ref, X_test_sROMs_fast, rtol=1e-5, atol=1e-7))
print("max diff:", diff.max(), "mean diff:", diff.mean())

same output? False
max diff: 0.5 mean diff: 0.0009890891482719332


**Faster GPU ideas:** (1) **float32** – used above. (2) **Batch RL**: if all snapshots share the same sigma, run Richardson–Lucy on a 3D stack (batched `gaussian_filter_gpu`) instead of a loop over 2D images. (3) **Fewer iterations**: reduce `num_iter` if 10 is more than needed. (4) **Precompute sigma**: pass `sigmaD=np.array([...])` so `get_sigma` isn’t called inside the loop. (5) **Async copy-back**: copy result to CPU in a stream while starting the next RL to hide transfer.